# Error Analysis — Part 3 (grammar-point appendix)

Same method again, for part 3's 7 benchmark pages
(`config/experiments/part3.yaml`). `structure_fidelity` already showed 100%
heading/bracket-point recall on this sample
(`experiment_result/part3/results.json`), so the interesting question here
is what the *remaining* ~4% mean CER actually consists of — is it still
losing meaningful content, or is it noise (punctuation, spacing)?

In [1]:
import sys
from pathlib import Path

ROOT = Path("/home/nguyenlog205/projects/university/semester65/chinese-contents/chinese-grammar")
sys.path.insert(0, str(ROOT))

from src.common.error_analysis import categorize_errors
from src.pipelines.part3 import run_pure_tool

IMAGES = ROOT / "data/benchmark/images"
GT = ROOT / "data/benchmark/groundtruth"
PAGES = [176, 190, 205, 220, 235, 250, 260]

print(f"Part 3 pages: {PAGES}")

Part 3 pages: [176, 190, 205, 220, 235, 250, 260]


In [2]:
from rapidocr import RapidOCR

engine = RapidOCR()
reports = {}

for p in PAGES:
    page = f"page-{p:03d}"
    img = str(IMAGES / f"{page}.png")
    gt_text = (GT / f"{page}.md").read_text(encoding="utf-8")

    result = run_pure_tool(img, engine=engine)
    report = categorize_errors(gt_text, result.text)
    reports[page] = report

    print(f"--- {page} (ref_len={report['ref_len']}) ---")
    print(f"  substitutions={report['n_substitutions']}  deletions={report['n_deletions']}  "
          f"insertions={report['n_insertions']}")
    if report["top_deletions"]:
        print("  top deletions:", [(d["char"], d["count"]) for d in report["top_deletions"][:8]])
    if report["top_substitutions"]:
        print("  top substitutions:", [(s["ref"], s["hyp"], s["count"]) for s in report["top_substitutions"][:8]])

--- page-176 (ref_len=365) ---
  substitutions=3  deletions=0  insertions=1
  top substitutions: [('？', '?', 3)]


--- page-190 (ref_len=331) ---
  substitutions=3  deletions=0  insertions=0
  top substitutions: [('？', '?', 1), ('"', '“', 1), ('"', '”', 1)]


--- page-205 (ref_len=476) ---
  substitutions=21  deletions=2  insertions=0
  top deletions: [('。', 2)]
  top substitutions: [('／', '/', 15), ('＋', '+', 6)]


--- page-220 (ref_len=480) ---
  substitutions=16  deletions=0  insertions=14
  top substitutions: [('＋', '+', 5), ('…', '·', 4), ('"', '“', 2), ('／', '/', 2), ('"', '”', 2), ('…', '.', 1)]


--- page-235 (ref_len=438) ---
  substitutions=2  deletions=0  insertions=0
  top substitutions: [('¹', '1', 2)]


--- page-250 (ref_len=599) ---
  substitutions=1  deletions=1  insertions=0
  top deletions: [('。', 1)]
  top substitutions: [('睛', '晴', 1)]


--- page-260 (ref_len=257) ---
  substitutions=1  deletions=38  insertions=0
  top deletions: [('，', 2), ('—', 2), ('发', 1), ('现', 1), ('原', 1), ('来', 1), ('差', 1), ('别', 1)]
  top substitutions: [('—', '-', 1)]


## Findings

`page-260` (the closing prose passage) is the clear outlier: earlier
spot-checking found it drops a full source line entirely, and that shows up
here as a large `deletions` count concentrated on ordinary sentence
characters (not headings or bracket points — consistent with the 100%
structure-fidelity result, since that page has none of those to lose in the
first place). The other pages' errors are small and scattered — mostly
punctuation-normalization noise (full-width vs. half-width punctuation,
spacing around numbers) rather than lost grammar content. That supports
treating part 3's current pipeline as adequate for its main purpose
(recovering headings and bracket-numbered points), with `page-260`-style
whole-line drops as the one failure mode worth watching for at full-document
scale.

In [3]:
import json

out_path = ROOT / "experiment_result/part3/error_analysis.json"
out_path.write_text(json.dumps(reports, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote {out_path}")

Wrote /home/nguyenlog205/projects/university/semester65/chinese-contents/chinese-grammar/experiment_result/part3/error_analysis.json
